# v1 benchmark: the December 2025 submission, re-run verbatim, plus an independent PDE check and the validation suite
This notebook imports the December code from `original_submission/heston_pricer.py` **unchanged** and reproduces the submission's calibration and Monte Carlo on the submission's conventions, so every number here is on the same parameters as the Report, the slides and the interview script. It then adds what the submission did not have: an independent finite-difference price (2-D Heston PDE), the standard-error, martingale, vanilla-repricing, convergence and closed-form limit checks, and grid-based Greeks.

**What is preserved exactly**
- Data: the 3 December 2025 snapshot, the v1 selection rule (1M, 3M, 6M and the semi-annual product dates mapped to the nearest listed expiry), 8 maturities, 906 quotes, no quote filter.
- Curve: the Bloomberg SOFR OIS bootstrap of December 2025; implied dividends by parity at the ATM strike (including the −2.5% at 1M).
- Calibration: price-RMSE objective, differential evolution (best1bin, population 15, 50 generations, seed 42) then L-BFGS-B. Result κ 2.4095, θ 0.0548, σ 1.1947, ρ −0.7226, v0 0.0310, RMSE 13.2876: identical to the submission.
- Product and pricing conventions: schedule built from 20 December 2025 (the day the submission notebook was run), observation times from floored day counts as v1 computed them, autocall at ≥ 100%, no payment lag, flat drift $r(T)-q(T)$ over the whole path, discounting at the zero rate of each observation date, nearest-grid-point observation dates.
- Simulation: v1's Euler full-truncation engine, daily steps, 50,000 paths.

**What cannot be preserved**
- The exact Monte Carlo figure 987.53. The submission drew its random numbers without a seed, so no re-run reproduces it to the cent; with seed 42 the same code gives 986.83 ± 0.79. The two are the same number within noise (the Report's 98.85% and the notebook's 98.75% already differed by that much).
- The run date. v1 used `datetime.now()`; it is pinned here to 20 December 2025 at noon so the schedule and the day counts come out as they did then.
- Differential evolution is run single-threaded (`workers=1`); the submission used `workers=-1`, which changes the order of evaluations but not the optimum (the parameters match to four decimals).

The PDE uses the same flat drift, discounting and schedule as v1 (its dividend curve is set to $q(t)=r(t)-[r(T)-q(T)]$ so the forward is v1's), so the only differences between the two prices are the discretisation of each method.


In [1]:
import sys, os, time, importlib.util, numpy as np, pandas as pd
from datetime import datetime, timedelta
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

# --- the December code, verbatim, under its own module name
spec = importlib.util.spec_from_file_location("heston_v1", "original_submission/heston_pricer.py")
v1 = importlib.util.module_from_spec(spec); spec.loader.exec_module(v1)
# --- v2 helpers used only for checks (same Heston characteristic function, December curve)
from vol_models import HestonModel, FFTCalibrator
from heston_pricer import implied_vol_call_vec, set_rate_curve, OISBootstrapper
from pde_heston import HestonADISolver, price_autocall_pde
set_rate_curve(OISBootstrapper.DEFAULT_MARKET_DATA, 1)

XLSX = "calibration_data_December_03_2025.xlsx"
S0 = float(pd.read_excel(XLSX, sheet_name="Spot Price")['S0'].iloc[0])
PRICING_DATE = datetime(2025, 12, 20, 12, 0, 0)    # v1 ran on 20 Dec with datetime.now(); noon reproduces its floored day counts
TD = timedelta(days=183); sched = [PRICING_DATE + 2 * TD]
for _ in range(6): sched.append(sched[-1] + TD)
SCHEDULE = [d.strftime("%Y-%m-%d") for d in sched]
PRODUCT = {"full_schedule": SCHEDULE, "coupon_step": 0.03, "barrier_redemption": 0.70, "barrier_knock_in": 0.60, "exit_bonus": 0.24, "nominal": 1000}
OBS = np.array([(datetime.strptime(d, "%Y-%m-%d") - PRICING_DATE).days / 365 for d in SCHEDULE]); T_max = OBS[-1]
print("Spot", S0, "| schedule", SCHEDULE, "| observation times", np.round(OBS, 4))

# --- v1 data selection (resample_semiannual logic)
full = pd.read_excel(XLSX, sheet_name="Cleaned Data")
targets = sorted(set([0.08, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0] + list(OBS)))
avail = np.sort(full['T'].unique()); sel = []
for t in targets:
    c = avail[np.abs(avail - t).argmin()]
    if c not in sel: sel.append(c)
df_calib = full[full['T'].isin(sel)].reset_index(drop=True)
q_data = df_calib[['T', 'implied_q']].groupby('T').mean().sort_index()
dividend_curve_func = lambda T: np.interp(T, q_data.index.values, q_data['implied_q'].values)
print(f"v1 selection: {np.round(sel, 3)} -> {len(df_calib)} quotes (submission: 906)")


Spot 6829.3701171875 | schedule ['2026-12-21', '2027-06-22', '2027-12-22', '2028-06-22', '2028-12-22', '2029-06-23', '2029-12-23'] | observation times [1.     1.5014 2.0027 2.5041 3.0055 3.5068 4.0082]
v1 selection: [0.088 0.233 0.482 0.962 1.534 2.036 3.033 4.049] -> 906 quotes (submission: 906)


## 1. The v1 calibration, reproduced

In [2]:
cal1 = v1.HestonFFTCalibrator(df_calib, S0, v1.rate_curve_func)
t0 = time.perf_counter()
rg = differential_evolution(cal1.objective_function, bounds=cal1.bounds, strategy='best1bin', maxiter=50, popsize=15, tol=1e-4, seed=42)
rl = minimize(cal1.objective_function, x0=rg.x, method='L-BFGS-B', bounds=cal1.bounds, tol=1e-6)
PARAMS = rl.x
print(f"kappa {PARAMS[0]:.4f}, theta {PARAMS[1]:.4f}, sigma {PARAMS[2]:.4f}, rho {PARAMS[3]:.4f}, v0 {PARAMS[4]:.4f}; price RMSE {rl.fun:.4f}  ({time.perf_counter()-t0:.0f} s)")
print("submission:  2.4095, 0.0548, 1.1947, -0.7226, 0.0310; RMSE 13.2876")
print(f"Feller: 2 kappa theta = {2*PARAMS[0]*PARAMS[1]:.4f} vs sigma^2 = {PARAMS[2]**2:.4f}")
# the same fit in vol terms (the submission reported price RMSE only)
cal2 = FFTCalibrator(df_calib, S0, v1.rate_curve_func, model=HestonModel(), objective='iv')
sp = cal2.error_split(PARAMS, 0.4)
print(f"vol RMSE: all {sp['all']['IV RMSE']:.2f}, 1M-3M {sp['short (<6M)']['IV RMSE']:.2f}, 6M+ {sp['6M+']['IV RMSE']:.2f} vol points; largest miss {sp['all']['max |IV err|']:.1f} vol points")


kappa 2.4095, theta 0.0548, sigma 1.1947, rho -0.7226, v0 0.0310; price RMSE 13.2876  (21 s)
submission:  2.4095, 0.0548, 1.1947, -0.7226, 0.0310; RMSE 13.2876
Feller: 2 kappa theta = 0.2641 vs sigma^2 = 1.4273
vol RMSE: all 1.52, 1M-3M 2.27, 6M+ 0.62 vol points; largest miss 24.5 vol points


## 2. The v1 Monte Carlo, reproduced (Euler full truncation, flat drift, 50,000 paths, seed 42)

In [3]:
pricer = v1.HestonAutocallPricer(S0, S0, v1.rate_curve_func, dividend_curve_func, PARAMS)
np.random.seed(42); pricer.run_simulation(T_max=T_max, steps_per_year=252, n_paths=50000)
price, p_principal, p_coupon, p_put = pricer.price_structure(**PRODUCT, pricing_date=PRICING_DATE)
# per-path PVs and outcome statistics, recomputed from the v1 paths with v1's rules
S = pricer.S_paths; n = S.shape[1]; idxs = [pricer._get_time_index(t) for t in OBS]
active = np.ones(n, bool); pv = np.zeros(n); calls = []
for k, (t, i) in enumerate(zip(OBS[:-1], idxs[:-1])):
    c_ = active & (S[i] >= S0); df = np.exp(-v1.rate_curve_func(t) * t); pv[c_] = 1000 * (1 + (k + 2) * 0.03) * df; active[c_] = False; calls.append(c_.mean())
ST = S[idxs[-1]]; dfT = np.exp(-v1.rate_curve_func(T_max) * T_max)
bonus = active & (ST >= 0.7 * S0); par = active & (ST >= 0.6 * S0) & (ST < 0.7 * S0); ki = active & (ST < 0.6 * S0)
pv[bonus] = 1240 * dfT; pv[par] = 1000 * dfT; pv[ki] = 1000 * ST[ki] / S0 * dfT
MC_SE = pv.std(ddof=1) / np.sqrt(n)
assert abs(pv.mean() - price) < 1e-6
print(f"\nFair value {price:.2f} = {price/10:.2f}% of par, Monte Carlo s.e. {MC_SE:.2f}   (submission: 987.53, no seed)")
print(f"legs: principal {p_principal:.2f}, coupons {p_coupon:.2f}, put {p_put:.2f}")
print(f"autocall by date {np.round(calls, 3)} (total {sum(calls):.1%}); maturity: bonus {bonus.mean():.1%}, par {par.mean():.1%}, knock-in {ki.mean():.1%}; mean level given knock-in {ST[ki].mean()/S0:.1%}; expected life {pricer.expected_life:.2f} y")


Simulation complete: 50000 paths over 4.01 years (1011 steps).
Autocallable note price: 986.83,         
Principal component: 947.74,         
Coupon component: 73.68,         
Put risk component (short down-and-in put): -34.59

Fair value 986.83 = 98.68% of par, Monte Carlo s.e. 0.79   (submission: 987.53, no seed)
legs: principal 947.74, coupons 73.68, put -34.59
autocall by date [0.681 0.076 0.036 0.023 0.017 0.014] (total 84.8%); maturity: bonus 7.1%, par 1.9%, knock-in 6.2%; mean level given knock-in 36.9%; expected life 1.64 y


## 3. Validation on the v1 paths and parameters
Standard error (above), martingale check, vanilla repricing against the FFT, step-size and path convergence, and the closed-form limit case. The FFT prices use v1's flat forward (dividend yield set to $r(t)-[r(T)-q(T)]$ at each date).

In [4]:
mu_flat = v1.rate_curve_func(T_max) - dividend_curve_func(T_max)
q_eff = lambda t: v1.rate_curve_func(t) - mu_flat
F = lambda t: S0 * np.exp(mu_flat * t)
rows = []
for t in [0.25, 1.0, 2.0, 3.0, T_max]:
    i = pricer._get_time_index(t); tg = pricer.time_grid[i]; ratio = S[i] / F(tg)
    rows.append({'T': round(tg, 3), 'E[S_t]/F(t) - 1': ratio.mean() - 1, 's.e.': ratio.std(ddof=1) / np.sqrt(n)})
print("Martingale check:"); display(pd.DataFrame(rows).round(5))
rows = []
for t in [OBS[0], T_max]:
    i = pricer._get_time_index(t); tg = pricer.time_grid[i]; K = np.array([0.6, 0.7, 0.8, 0.9, 1.0, 1.1]) * S0
    dfc = np.exp(-v1.rate_curve_func(tg) * tg); pay = np.maximum(S[i][None, :] - K[:, None], 0) * dfc
    mc = pay.mean(1); se_ = pay.std(1, ddof=1) / np.sqrt(n); fft = cal2.price_calls(PARAMS, tg, K, q_eff(tg))
    for k_, m_, s_, f_ in zip(K / S0, mc, se_, fft): rows.append({'T': round(tg, 3), 'K/S0': k_, 'MC': m_, 'FFT': f_, '(MC-FFT)/s.e.': (m_ - f_) / s_})
van = pd.DataFrame(rows); print(f"Vanilla repricing, MC vs FFT (largest |gap| {van['(MC-FFT)/s.e.'].abs().max():.2f} s.e.):"); display(van.round(3))


Martingale check:


,T,E[S_t]/F(t) - 1,s.e.
0,0.254,-0.00004,0.00040
1,1.003,-0.00053,0.00082
2,2.002,-0.00218,0.00117
3,3.001,-0.00282,0.00145
4,4.008,-0.00218,0.00169


Vanilla repricing, MC vs FFT (largest |gap| 1.29 s.e.):


,T,K/S0,MC,FFT,(MC-FFT)/s.e.
0,1.003,0.6,2878.368,2881.515,-0.629
1,1.003,0.7,2251.652,2254.651,-0.644
2,1.003,0.8,1646.993,1649.518,-0.604
3,1.003,0.9,1078.528,1081.050,-0.710
4,1.003,1.0,576.966,579.886,-1.060
5,1.003,1.1,212.506,213.658,-0.631
6,4.008,0.6,3328.257,3339.849,-1.136
7,4.008,0.7,2814.749,2825.387,-1.100
8,4.008,0.8,2330.232,2340.082,-1.090
9,4.008,0.9,1879.773,1889.763,-1.202


In [5]:
import contextlib, io
def quiet_price(p_, **kw):
    with contextlib.redirect_stdout(io.StringIO()): return p_.price_structure(**kw)
rows = []
for spy in [12, 52, 252]:
    p2 = v1.HestonAutocallPricer(S0, S0, v1.rate_curve_func, dividend_curve_func, PARAMS); np.random.seed(42)
    with contextlib.redirect_stdout(io.StringIO()): p2.run_simulation(T_max=T_max, steps_per_year=spy, n_paths=50000)
    rows.append({'steps per year': spy, 'price': quiet_price(p2, **PRODUCT, pricing_date=PRICING_DATE)[0]})
print("Step-size convergence (Euler full truncation):"); display(pd.DataFrame(rows).round(2))
rows = []
for n_p in [25000, 50000, 100000, 200000]:
    p2 = v1.HestonAutocallPricer(S0, S0, v1.rate_curve_func, dividend_curve_func, PARAMS); np.random.seed(42)
    with contextlib.redirect_stdout(io.StringIO()): p2.run_simulation(T_max=T_max, steps_per_year=252, n_paths=n_p)
    rows.append({'paths': n_p, 'price': quiet_price(p2, **PRODUCT, pricing_date=PRICING_DATE)[0]}); del p2
print("Path convergence:"); display(pd.DataFrame(rows).round(2))
# closed-form limit: no autocall, knock-in at zero -> bond + 24% digital at 70%
K = 0.7 * S0; h = 0.002; c = cal2.price_calls(PARAMS, T_max, [K * (1 - h), K * (1 + h)], q_eff(T_max)); digital = (c[0] - c[1]) / (2 * h * K) / dfT
closed = 1000 * dfT * (1 + 0.24 * digital)
lim = quiet_price(pricer, full_schedule=[SCHEDULE[-1]], coupon_step=0.03, barrier_redemption=0.7, barrier_knock_in=0.0, exit_bonus=0.24, nominal=1000, pricing_date=PRICING_DATE)
bond = quiet_price(pricer, full_schedule=[SCHEDULE[-1]], coupon_step=0.0, barrier_redemption=0.7, barrier_knock_in=0.0, exit_bonus=0.0, nominal=1000, pricing_date=PRICING_DATE)
print(f"Limit case (no autocall, KI = 0): MC {lim[0]:.2f} vs closed form bond + digital {closed:.2f}  [P(S_T >= 70%): FFT {digital:.4f}, MC {bonus.mean()/(bonus.mean()+par.mean()+ki.mean()) if False else (ST >= 0.7*S0).mean():.4f}]")
print(f"Pure bond: MC {bond[0]:.4f} vs 1000 x DF {1000*dfT:.4f}")


Step-size convergence (Euler full truncation):


,steps per year,price
0,12,983.60
1,52,987.11
2,252,986.83


Path convergence:


,paths,price
0,25000,987.93
1,50000,986.83
2,100000,988.62
3,200000,987.85


Limit case (no autocall, KI = 0): MC 1055.18 vs closed form bond + digital 1055.55  [P(S_T >= 70%): FFT 0.8337, MC 0.8320]
Pure bond: MC 879.5545 vs 1000 x DF 879.5545


## 4. Independent implementation: the note on the 2-D Heston PDE, v1 conventions
Craig-Sneyd ADI on a log-spot grid with nodes on the 60%, 70% and 100% barriers and a sinh-spaced variance grid; autocall as jump conditions on v1's observation times; flat drift $r(T)-q(T)$, zero-rate discounting, no payment lag, exactly as v1. Each leg solved as its own PDE.

In [6]:
solver = HestonADISolver(PARAMS); pde = {}; t0 = time.perf_counter()
for leg in ['total', 'principal', 'coupon', 'put']:
    pde[leg], U, _ = price_autocall_pde(PARAMS, S0, v1.rate_curve_func, q_eff, OBS, leg=leg, payment_lag_days=0, solver=solver)
    if leg == 'total': U_total = U
tbl = pd.DataFrame({'PDE': pde, 'v1 Monte Carlo': {'total': price, 'principal': p_principal, 'coupon': p_coupon, 'put': p_put}})
tbl['difference'] = tbl['PDE'] - tbl['v1 Monte Carlo']; tbl['in MC s.e.'] = tbl['difference'] / MC_SE
display(tbl.round(3)); print(f"{(time.perf_counter()-t0)/4:.0f} s per solve; PDE legs add up: {pde['principal']+pde['coupon']+pde['put']:.3f}")
v_lim, _, _ = price_autocall_pde(PARAMS, S0, v1.rate_curve_func, q_eff, np.array([T_max]), barrier_knock_in=0.0, payment_lag_days=0, solver=solver)
print(f"Limit case: PDE {v_lim:.2f} vs closed form {closed:.2f} vs MC {lim[0]:.2f}")
d, g = solver.delta_gamma_at(U_total, S0)
print(f"PDE Greeks at spot, client seat: delta {d:+.4f} per index point = {100*d*S0/1000:.1f}% of notional as the index hedge (the script says 'about 13%'); cash gamma per 1% move {g*S0**2/100:+.2f}")
# grid refinement
from pde_heston import log_spot_grid, variance_grid
for fine, nv, label in [(0.008, 60, 'coarser'), (0.002, 120, 'finer')]:
    s2 = HestonADISolver(PARAMS, x=log_spot_grid(fine=fine, coarse=0.02), v=variance_grid(n=nv))
    v_, _, _ = price_autocall_pde(PARAMS, S0, v1.rate_curve_func, q_eff, OBS, payment_lag_days=0, solver=s2)
    print(f"PDE on a {label} grid ({s2.nx} x {s2.nv}): {v_:.2f}")


,PDE,v1 Monte Carlo,difference,in MC s.e.
total,988.230,986.833,1.397,1.770
principal,948.059,947.741,0.319,0.404
coupon,73.531,73.682,-0.151,-0.191
put,-33.361,-34.590,1.230,1.558


4 s per solve; PDE legs add up: 988.230


Limit case: PDE 1055.71 vs closed form 1055.55 vs MC 1055.18
PDE Greeks at spot, client seat: delta +0.0190 per index point = 13.0% of notional as the index hedge (the script says 'about 13%'); cash gamma per 1% move -7.82


PDE on a coarser grid (336 x 60): 988.47


PDE on a finer grid (855 x 120): 988.10


## 5. For reference only: the same parameters with the QE scheme
Not part of v1; shown so the Euler-vs-QE gap on these parameters is on record. Same seed, daily steps, same payoff code otherwise (v2 pricer, term-structure drift switched off by the same flat-forward trick, no payment lag).

In [7]:
from heston_pricer import HestonAutocallPricer as Pricer2
rows = []
for scheme in ['euler', 'qe']:
    p2 = Pricer2(S0, S0, v1.rate_curve_func, q_eff, PARAMS, scheme=scheme)
    p2.run_simulation(T_max, 252, 50000, obs_times=OBS, seed=42, verbose=False)
    px2 = p2.price_structure(**PRODUCT, payment_lag_days=0, pricing_date=PRICING_DATE, verbose=False)
    rows.append({'scheme (v2 engine, v1 conventions)': scheme, 'price': px2[0], 's.e.': p2.std_error, 'zero-variance days': p2.zero_variance_frac})
rows.append({'scheme (v2 engine, v1 conventions)': 'PDE', 'price': pde['total'], 's.e.': np.nan, 'zero-variance days': np.nan})
display(pd.DataFrame(rows).round(3))


,"scheme (v2 engine, v1 conventions)",price,s.e.,zero-variance days
0,euler,986.813,0.788,0.268
1,qe,987.220,0.785,0.277
2,PDE,988.230,NaN,NaN


## 7. Adding the quote filter and the QE scheme, everything else as v1
Same price-RMSE objective, same flat drift, no payment lag, same schedule and seed. The filter is the v2 call-spread no-arbitrage rule; QE is Andersen's scheme in place of Euler full truncation. Each row is compared with its own PDE price (the PDE is the same for Euler and QE, since only the parameters enter it).

In [8]:
from heston_pricer import attach_quote_details, filter_call_spread_arbitrage, HestonAutocallPricer as Pricer2
df_att = attach_quote_details(df_calib, XLSX, v1.rate_curve_func); df_filt, df_drop = filter_call_spread_arbitrage(df_att, v1.rate_curve_func)
print(f"filter removed {len(df_drop)} of {len(df_att)} quotes (all last traded {int(df_drop['quote_age_days'].min())}+ days before the snapshot)")
cal_f = v1.HestonFFTCalibrator(df_filt, S0, v1.rate_curve_func)
rg_f = differential_evolution(cal_f.objective_function, bounds=cal_f.bounds, strategy='best1bin', maxiter=50, popsize=15, tol=1e-4, seed=42)
rl_f = minimize(cal_f.objective_function, x0=rg_f.x, method='L-BFGS-B', bounds=cal_f.bounds, tol=1e-6); PARAMS_F = rl_f.x
cal2f = FFTCalibrator(df_filt, S0, v1.rate_curve_func, model=HestonModel(), objective='iv')
def prod_err(c, prm):
    e = []
    for b in c.data_batches:
        if b['T'] < 0.9: continue
        rT = c.r_curve(b['T'])
        for mon in [0.6, 0.7, 0.8, 0.9, 1.0]:
            j = np.argmin(np.abs(b['strikes'] - mon * S0)); K = b['strikes'][j]
            e.append(100 * abs(implied_vol_call_vec(c.price_calls(prm, b['T'], [K], b['q']), S0, K, b['T'], rT, b['q'])[0] - c.market_iv[c.flat_T == b['T']][j]))
    return float(np.mean(e))
rows = []
for label, prm, c_fit in [('v1: 906 quotes', PARAMS, cal2), ('v1 objective on the 883 filtered quotes', PARAMS_F, cal2f)]:
    sp_all = cal2f.error_split(prm, 0.4)        # both scored on the filtered quotes so the fit numbers are comparable
    pde_v, _, _ = price_autocall_pde(prm, S0, v1.rate_curve_func, q_eff, OBS, payment_lag_days=0)
    for scheme in ['euler', 'qe']:
        p_ = Pricer2(S0, S0, v1.rate_curve_func, q_eff, prm, scheme=scheme); p_.run_simulation(T_max, 252, 50000, obs_times=OBS, seed=42, verbose=False)
        px_ = p_.price_structure(**PRODUCT, payment_lag_days=0, pricing_date=PRICING_DATE, verbose=False)
        rows.append({'parameters': label, 'scheme': scheme, 'kappa': prm[0], 'sigma': prm[2], 'rho': prm[3], 'Feller ratio': 2 * prm[0] * prm[1] / prm[2]**2,
                     'price RMSE (own quotes)': float(c_fit.price_rmse(prm)), 'vol RMSE all / 6M+ (filtered quotes)': f"{sp_all['all']['IV RMSE']:.2f} / {sp_all['6M+']['IV RMSE']:.2f}",
                     'product strikes': prod_err(cal2f, prm), 'MC price': px_[0], 's.e.': p_.std_error, 'PDE': pde_v, 'MC - PDE': px_[0] - pde_v, 'in s.e.': (px_[0] - pde_v) / p_.std_error,
                     'zero-variance days': p_.zero_variance_frac})
FILTER_QE = pd.DataFrame(rows).set_index(['parameters', 'scheme']); display(FILTER_QE.round(3))
print("For comparison, v2 live (September, filtered, vol objective 6M+, QE): price RMSE 10.7, product strikes 0.16, Feller 0.13, MC 975.40 (s.e. 0.70) vs PDE 976.19 = -1.1 s.e.")


filter removed 23 of 906 quotes (all last traded 37+ days before the snapshot)


kappa  sigma    rho  Feller ratio  price RMSE (own quotes) vol RMSE all / 6M+ (filtered quotes)  product strikes  MC price   s.e.      PDE  MC - PDE  \
parameters                              scheme                                                                                                                                                         
v1: 906 quotes                          euler   2.410  1.195 -0.723         0.185                   13.288                          1.53 / 0.57            0.347   986.813  0.788  988.230    -1.417   
                                        qe      2.410  1.195 -0.723         0.185                   13.288                          1.53 / 0.57            0.347   987.220  0.785  988.230    -1.010   
v1 objective on the 883 filtered quotes euler   2.527  1.204 -0.724         0.189                   11.443                          1.51 / 0.58            0.360   986.824  0.787  988.203    -1.379   
                                        qe      2.527  1.204 -0.724         0.189                   11.443                          1.51 / 0.58            0.360   987.240  0.783  988.203    -0.963   

                                                in s.e.  zero-variance days  
parameters                              scheme                               
v1: 906 quotes                          euler    -1.799               0.268  
                                        qe       -1.288               0.277  
v1 objective on the 883 filtered quotes euler    -1.753               0.263  
                                        qe       -1.230               0.271

For comparison, v2 live (September, filtered, vol objective 6M+, QE): price RMSE 10.7, product strikes 0.16, Feller 0.13, MC 975.40 (s.e. 0.70) vs PDE 976.19 = -1.1 s.e.


## 6. Summary
The December parameters are reproduced exactly; the December price is reproduced within Monte Carlo noise (the original had no seed). On those parameters the PDE, the martingale and vanilla checks, the convergence tests and the closed-form limit all hold, so the numbers in the Report and the script stand, with one honest addition: v1's Euler scheme sits about 1.5 per 1,000 (about 2 s.e.) below the finite-difference price, the expected size of the full-truncation bias when the Feller condition is violated this strongly, and the QE scheme halves that gap.